In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mtcnn import MTCNN
from tqdm import tqdm
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

class FaceProcessor:
    def __init__(self):
        self.detector = MTCNN()

    def align_and_crop(self, img, output_size=(224, 224)):
        if img is None: return None
        
        h, w = img.shape[:2]
        if h < 80 or w < 80:
            img = cv2.resize(img, (160, 160), interpolation=cv2.INTER_CUBIC)
            h, w = img.shape[:2]

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        try:
            results = self.detector.detect_faces(img_rgb)
        except Exception as e:
            return None
        
        if not results:
            return None

        face = results[0]
        keypoints = face['keypoints']
        le, re = keypoints['left_eye'], keypoints['right_eye']

        dY = re[1] - le[1]
        dX = re[0] - le[0]
        angle = np.degrees(np.arctan2(dY, dX))
        center = (float((le[0] + re[0]) / 2), float((le[1] + re[1]) / 2))
        
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated_img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC)

        x, y, bw, bh = face['box']
        margin = 30
        y1, y2 = max(0, y - margin), min(h, y + bh + margin)
        x1, x2 = max(0, x - margin), min(w, x + bw + margin)
        
        crop = rotated_img[y1:y2, x1:x2]
        
        if crop.size == 0: return None
        return cv2.resize(crop, output_size)

    def process_folders(self, input_base, output_base):
        categories = ['real', 'fake']
        total_success, total_fail = 0, 0
        
        for cat in categories:
            input_path = os.path.join(input_base, cat)
            if not os.path.exists(input_path):
                input_path = os.path.join(input_base, cat.upper())
                
            output_path = os.path.join(output_base, cat.lower())
            os.makedirs(output_path, exist_ok=True)

            if not os.path.exists(input_path):
                continue

            print(f"\nSTARTING BATCH: {cat.upper()} images")
            files = [f for f in os.listdir(input_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
            
            cat_success = 0
            for filename in tqdm(files, desc=f"Aligning {cat}"):
                try:
                    img_path = os.path.join(input_path, filename)
                    img = cv2.imread(img_path)
                    
                    processed = self.align_and_crop(img)
                    
                    if processed is not None:
                        cv2.imwrite(os.path.join(output_path, filename), processed)
                        cat_success += 1
                    else:
                        with open("failed_images.log", "a") as log:
                            log.write(f"SKIPPED (No Face): {cat}/{filename}\n")
                except Exception:
                    continue
            
            print(f"✅ Finished {cat}: {cat_success}/{len(files)} saved.")
            total_success += cat_success
            total_fail += (len(files) - cat_success)

        print("\n" + "="*30)
        print(f"PHASE 1 FINAL SUMMARY")
        print(f"Total Aligned: {total_success}")
        print(f"Total Skipped: {total_fail}")
        print("="*30)

# Configuration
BASE_DIR = "D:/VS Code/Coding/Deepfake_Project/Phase1_Preprocessing"
RAW_DIR = os.path.join(BASE_DIR, "raw_data")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_data")

processor = FaceProcessor()
processor.process_folders(RAW_DIR, PROCESSED_DIR)

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "d:\VS Code\Coding\Deepfake_Project\.venv\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "d:\VS Code\Coding\Deepfake_Project\.venv\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "d:\VS Code\Coding\Deepfake_Project\.venv\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.



STARTING BATCH: REAL images


Aligning real: 100%|██████████| 70001/70001 [2:01:54<00:00,  9.57it/s]  


✅ Finished real: 69277/70001 saved.

STARTING BATCH: FAKE images


Aligning fake: 100%|██████████| 70001/70001 [2:10:54<00:00,  8.91it/s]  

✅ Finished fake: 69361/70001 saved.

PHASE 1 FINAL SUMMARY
Total Aligned: 138638
Total Skipped: 1364
